# 09 — Industry-Grade Methods

Standard z-tests and p-values get you to a correct answer — but at scale, "correct"
is not enough. Big tech experimentation platforms layer on three key methods that make
experiments faster, more accurate, and safer to monitor continuously.

| Method | Problem it solves | Used by |
|---|---|---|
| **CUPED** | Noisy metrics require huge samples | LinkedIn, Netflix, Booking.com, Spotify |
| **Delta Method** | Ratio metrics have wrong variance under standard tests | Meta, Google, Airbnb |
| **Sequential Testing** | Can't safely peek at results mid-experiment | Spotify, Airbnb, Statsig, Eppo |

Each section below: (1) the problem, (2) the math, (3) implementation on our dataset,
(4) comparison to the standard approach, (5) business commentary.

In [ ]:
import sys, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from scipy import stats

sys.path.append("../src")
from ab_stats import (
    two_proportion_ztest, confidence_interval,
    cuped_adjust, delta_method_ratio_test, obrien_fleming_boundary
)

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120
np.random.seed(42)
ALPHA = 0.05

df = pd.read_csv("../data/ab_data_clean.csv")
ctrl_df = df[df["group"] == "control"].copy()
trt_df  = df[df["group"] == "treatment"].copy()
print(f"Control: {len(ctrl_df):,} | Treatment: {len(trt_df):,}")

---
## Method 1: CUPED
### The Problem

Conversion rate is noisy. Users vary enormously in their engagement, intent, and
likelihood to convert — and that variation swamps the signal from the treatment.
To detect a small effect, you need a large sample. Large samples mean longer experiments.
Longer experiments mean slower decisions.

**The insight:** If you can explain some of that pre-existing variance using data you
already have, the residual variance is smaller — and you need fewer observations.

### The Math

CUPED adjusts the outcome Y using a pre-experiment covariate X:

$$Y_{\text{cuped}} = Y - \theta (X - \mathbb{E}[X])$$

where $\theta = \frac{\text{Cov}(Y, X)}{\text{Var}(X)}$ is chosen to minimise variance.

The adjusted outcome has the same expectation as Y (the adjustment is mean-zero),
but lower variance:

$$\text{Var}(Y_{\text{cuped}}) = \text{Var}(Y)(1 - \rho^2)$$

where $\rho$ is the correlation between Y and X. A covariate with $\rho = 0.5$
reduces variance by 25%, equivalent to getting 33% more data for free.

### What counts as a good covariate X?

- Must be measured **before** treatment assignment (pre-experiment)
- Must be **correlated** with the outcome Y
- Common choices: prior conversion rate, prior session count, prior spend

> **Note:** Our dataset does not include true pre-experiment data, so we simulate
> a realistic prior engagement score correlated with conversion — this mirrors what
> companies like LinkedIn and Booking.com do with their user history.

In [ ]:
# Simulate pre-experiment engagement score (prior visit behaviour)
# Converted users tend to have higher prior engagement — a realistic assumption
def simulate_prior_engagement(converted, noise_scale=2.5, seed=0):
    rng = np.random.default_rng(seed)
    signal = converted.values * 3.0
    noise  = rng.gamma(shape=3, scale=noise_scale, size=len(converted))
    prior  = signal + noise
    return (prior - prior.mean()) / prior.std()   # standardise

ctrl_df["prior_engagement"] = simulate_prior_engagement(ctrl_df["converted"], seed=1)
trt_df["prior_engagement"]  = simulate_prior_engagement(trt_df["converted"],  seed=2)

rho_ctrl = np.corrcoef(ctrl_df["converted"], ctrl_df["prior_engagement"])[0, 1]
rho_trt  = np.corrcoef(trt_df["converted"],  trt_df["prior_engagement"])[0, 1]
print(f"Correlation (control):   rho = {rho_ctrl:.3f}")
print(f"Correlation (treatment): rho = {rho_trt:.3f}")
print(f"Expected variance reduction: ~{(1 - rho_ctrl**2)*100:.1f}% of original")

In [ ]:
# Apply CUPED adjustment
y_ctrl_raw = ctrl_df["converted"].values.astype(float)
y_trt_raw  = trt_df["converted"].values.astype(float)
x_ctrl     = ctrl_df["prior_engagement"].values
x_trt      = trt_df["prior_engagement"].values

# Compute theta using the POOLED covariate (standard practice)
x_all = np.concatenate([x_ctrl, x_trt])
y_all = np.concatenate([y_ctrl_raw, y_trt_raw])
theta = np.cov(y_all, x_all)[0, 1] / np.var(x_all)

y_ctrl_adj = y_ctrl_raw - theta * (x_ctrl - x_all.mean())
y_trt_adj  = y_trt_raw  - theta * (x_trt  - x_all.mean())

var_reduction = (1 - np.var(y_ctrl_adj) / np.var(y_ctrl_raw)) * 100
print(f"theta (adjustment coefficient): {theta:.4f}")
print(f"Variance reduction:             {var_reduction:.1f}%")

In [ ]:
# Compare: standard z-test vs CUPED z-test
z_raw,  p_raw,  pc_raw,  pt_raw,  lift_raw,  _ = two_proportion_ztest(
    int(y_ctrl_raw.sum()), len(y_ctrl_raw),
    int(y_trt_raw.sum()),  len(y_trt_raw)
)

# For CUPED, use a t-test on the adjusted continuous values
t_adj, p_adj = stats.ttest_ind(y_trt_adj, y_ctrl_adj)
lift_adj = y_trt_adj.mean() - y_ctrl_adj.mean()

se_raw = (pt_raw * (1-pt_raw) / len(y_trt_raw) + pc_raw * (1-pc_raw) / len(y_ctrl_raw)) ** 0.5
se_adj = (np.var(y_trt_adj, ddof=1)/len(y_trt_adj) + np.var(y_ctrl_adj, ddof=1)/len(y_ctrl_adj)) ** 0.5

print("=" * 55)
print("{:<22} {:>10} {:>10} {:>10}".format("Method", "Lift", "SE", "p-value"))
print("-" * 55)
print("{:<22} {:>+10.4%} {:>10.6f} {:>10.4f}".format("Standard z-test", lift_raw, se_raw, p_raw))
print("{:<22} {:>+10.4%} {:>10.6f} {:>10.4f}".format("CUPED", lift_adj, se_adj, p_adj))
print("=" * 55)
print(f"  SE reduction: {(1 - se_adj/se_raw)*100:.1f}%")
print(f"  Equivalent to {1/(1 - var_reduction/100):.2f}x more data")

In [ ]:
# Visualise: how much sample size CUPED saves across correlation levels
rhos = np.linspace(0, 0.8, 100)
n_reduction = (1 - rhos**2) * 100
equiv_data  = 1 / (1 - rhos**2)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

ax1.plot(rhos, n_reduction, color="steelblue", linewidth=2)
ax1.axvline(abs(rho_ctrl), color="tomato", linestyle="--",
            label=f"Our covariate (rho={rho_ctrl:.2f})")
ax1.axhline(var_reduction, color="tomato", linestyle=":")
ax1.set_xlabel("Correlation between covariate and outcome")
ax1.set_ylabel("Variance reduction (%)")
ax1.set_title("CUPED variance reduction vs covariate quality")
ax1.legend()

ax2.plot(rhos, equiv_data, color="coral", linewidth=2)
ax2.axvline(abs(rho_ctrl), color="steelblue", linestyle="--",
            label=f"Our covariate (rho={rho_ctrl:.2f})")
ax2.set_xlabel("Correlation between covariate and outcome")
ax2.set_ylabel("Equivalent data multiplier")
ax2.set_title("CUPED: effective data gain")
ax2.legend()

plt.suptitle("The value of a good pre-experiment covariate", fontsize=12)
plt.tight_layout()
plt.show()

### Business Commentary

- **LinkedIn** published the original CUPED paper (Deng et al., 2013) and reported
  that CUPED reduced experiment duration by ~50% across their platform.
- **Booking.com** applies CUPED to almost every experiment using prior booking history.
- **Netflix** uses stratification + CUPED for engagement metrics (watch time, completion rate).

**The practical limit:** variance reduction is bounded by the covariate's predictive power.
A rho of 0.4–0.6 (common for prior behaviour vs current behaviour) gives 16–36%
variance reduction. Beyond that, you need more informative covariates or stratification.

**When CUPED doesn't help:** when you have no pre-experiment data (new product, new users)
or when the outcome is completely unpredictable from prior behaviour.

---
## Method 2: Delta Method for Ratio Metrics
### The Problem

Most interesting business metrics are **ratios**:
- Revenue per user = total revenue / number of users
- Click-through rate = clicks / impressions
- Pages per visit = total pages viewed / number of visits

The standard z-test assumes your metric is a simple mean. When the metric is a ratio
where **both numerator and denominator vary per user**, the standard error formula
is wrong — it ignores the variance of the denominator and the covariance between them.

### The Math

For ratio $R = \bar{Y} / \bar{X}$, the delta method (first-order Taylor expansion) gives:

$$\text{Var}(R) \approx \frac{1}{n} \left[ \frac{\text{Var}(Y)}{\mu_X^2} - \frac{2R \cdot \text{Cov}(Y,X)}{\mu_X^2} + \frac{R^2 \cdot \text{Var}(X)}{\mu_X^2} \right]$$

The key terms:
- $\text{Var}(Y)/\mu_X^2$ — variance from the numerator
- $R^2 \text{Var}(X)/\mu_X^2$ — variance from the denominator
- $-2R \cdot \text{Cov}(Y,X)/\mu_X^2$ — covariance correction (often negative → inflates variance)

### Our metric: revenue per page visited

We compute `purchase_amount / pages_visited` per user — a genuine ratio metric
where both numerator (revenue) and denominator (pages) vary independently.
This represents revenue efficiency: how much revenue is generated per page a user visits.

In [ ]:
# Define the ratio metric: revenue per page visited
# Y = purchase_amount (per user), X = pages_visited (per user)
y_c = ctrl_df["purchase_amount"].values
x_c = ctrl_df["pages_visited"].values
y_t = trt_df["purchase_amount"].values
x_t = trt_df["pages_visited"].values

R_c_naive = y_c.mean() / x_c.mean()
R_t_naive = y_t.mean() / x_t.mean()

print(f"Control   — mean revenue: ${y_c.mean():.4f} | mean pages: {x_c.mean():.2f} | ratio: ${R_c_naive:.4f}/page")
print(f"Treatment — mean revenue: ${y_t.mean():.4f} | mean pages: {x_t.mean():.2f} | ratio: ${R_t_naive:.4f}/page")

In [ ]:
# Naive SE: treat ratio as a simple mean (WRONG for ratio metrics)
n_c, n_t = len(y_c), len(y_t)
se_naive_c = np.std(y_c / x_c.mean(), ddof=1) / np.sqrt(n_c)
se_naive_t = np.std(y_t / x_t.mean(), ddof=1) / np.sqrt(n_t)
se_naive   = np.sqrt(se_naive_c**2 + se_naive_t**2)

# Delta method SE (CORRECT)
R_c, R_t, diff, se_delta, z_d, p_d, ci_lo, ci_hi = delta_method_ratio_test(y_c, x_c, y_t, x_t)
z_naive = (R_t_naive - R_c_naive) / se_naive
p_naive = 2 * (1 - stats.norm.cdf(abs(z_naive)))

z_crit = stats.norm.ppf(1 - ALPHA / 2)
diff_naive = R_t_naive - R_c_naive
ci_naive_lo = diff_naive - z_crit * se_naive
ci_naive_hi = diff_naive + z_crit * se_naive

print("=" * 62)
print("{:<24} {:>10} {:>8} {:>8}".format("Method", "SE", "z", "p"))
print("-" * 62)
print("{:<24} {:>10.6f} {:>8.3f} {:>8.4f}".format("Naive (incorrect)", se_naive, z_naive, p_naive))
print("{:<24} {:>10.6f} {:>8.3f} {:>8.4f}".format("Delta method (correct)", se_delta, z_d, p_d))
print("=" * 62)

fig, ax = plt.subplots(figsize=(9, 3.5))
methods = ["Naive", "Delta method"]
los = [ci_naive_lo, ci_lo]
his = [ci_naive_hi, ci_hi]
diffs = [diff_naive, diff]
colors = ["tomato", "steelblue"]
for i, (m, lo, hi, d, c) in enumerate(zip(methods, los, his, diffs, colors)):
    ax.errorbar(d, i, xerr=[[d-lo],[hi-d]], fmt="o", color=c,
                capsize=8, markersize=9, linewidth=2.5, label=m)
ax.axvline(0, color="gray", linewidth=1.5, linestyle="--", label="No difference")
ax.set_yticks([0, 1])
ax.set_yticklabels(methods)
ax.set_xlabel("Difference in revenue per page visited ($/page)")
ax.set_title("95% CI on ratio metric: naive vs delta method")
ax.legend()
plt.tight_layout()
plt.show()

print(f"CI width — Naive: {ci_naive_hi-ci_naive_lo:.6f} | Delta: {ci_hi-ci_lo:.6f}")
print("The delta method CI is wider — the naive approach underestimates uncertainty.")

### Business Commentary

- **Meta** uses the delta method for all ratio metrics in their internal experimentation
  platform (XP). CTR, revenue per impression, and video completion rate all require it.
- **Airbnb** applies the delta method for metrics like revenue per night booked,
  where both bookings and revenue vary per host.
- **Google** extends it further with the "regression-adjusted" estimator for complex metrics.

**When the naive approach misleads you:**
- If Cov(Y, X) > 0 (revenue and pages visited are positively correlated — likely),
  the delta method SE is **larger** than naive. Naive overstates confidence.
- In high-traffic situations the difference is small, but at moderate N it can change
  whether a result clears the significance threshold.

**Rule of thumb:** if your metric is not a simple per-user mean, use the delta method.
The extra code is minimal; the cost of getting it wrong is a false positive.

---
## Method 3: Sequential Testing
### The Problem: The Peeking Trap

In notebook 01 we established that you commit to an end date before launch.
In practice, this almost never happens. A product manager asks "how's it looking?"
every day. Engineering wants to know if they can stop early. Leadership wants a result
before the quarterly review.

Checking results repeatedly and stopping when p < 0.05 is called **peeking**.
It inflates the Type I error rate dramatically — with daily checks over 4 weeks,
the effective false positive rate can reach 25–40% even though you set α = 0.05.

**Sequential testing solves this without sacrificing the ability to monitor continuously.**

### Two Approaches

**O'Brien-Fleming alpha spending:** Set a tighter threshold early in the experiment
that relaxes as you approach the planned end. You "spend" your alpha budget
conservatively at first, saving most of it for the final look.

$$z_{\text{boundary}}(t) = \frac{z_{\alpha/2}}{\sqrt{t}}$$

where $t$ is the information fraction (users enrolled so far / total planned).

**mSPRT (mixture Sequential Probability Ratio Test):** A test martingale that stays
valid at every sample size simultaneously. Used by Spotify and the "always valid
inference" paper (Johari et al., 2022). Reject when the likelihood ratio exceeds 1/α.

In [ ]:
# Simulate peeking: check significance every day for 28 days
# Under the null (both groups have the same true conversion rate)
N_DAYS   = 28
N_SIM    = 5_000
DAILY_N  = len(df) // N_DAYS     # users per day
TRUE_P   = df["converted"].mean() # null rate for both groups

naive_fp    = 0  # false positives under naive daily peeking
obf_fp      = 0  # false positives with O'Brien-Fleming boundaries

for _ in range(N_SIM):
    c_conv = t_conv = c_tot = t_tot = 0
    naive_rejected = obf_rejected = False

    for day in range(1, N_DAYS + 1):
        n_day = DAILY_N // 2
        c_conv += np.random.binomial(n_day, TRUE_P)
        t_conv += np.random.binomial(n_day, TRUE_P)
        c_tot  += n_day
        t_tot  += n_day

        p_pool = (c_conv + t_conv) / (c_tot + t_tot)
        se = np.sqrt(p_pool*(1-p_pool)*(1/c_tot + 1/t_tot))
        if se == 0: continue
        z = ((t_conv/t_tot) - (c_conv/c_tot)) / se

        t_frac = day / N_DAYS
        obf_boundary = obrien_fleming_boundary(t_frac, ALPHA)

        if abs(z) > stats.norm.ppf(1 - ALPHA/2): naive_rejected = True
        if abs(z) > obf_boundary: obf_rejected = True

    naive_fp += naive_rejected
    obf_fp   += obf_rejected

print(f"Nominal alpha:               {ALPHA:.0%}")
print(f"Naive peeking false pos rate: {naive_fp/N_SIM:.1%}  <- inflated!")
print(f"O'Brien-Fleming false pos rate:{obf_fp/N_SIM:.1%}  <- controlled")

In [ ]:
# Visualise O'Brien-Fleming boundaries vs naive threshold
t_fracs = np.linspace(0.05, 1.0, 100)
obf_bounds = [obrien_fleming_boundary(t, ALPHA) for t in t_fracs]
naive_bound = stats.norm.ppf(1 - ALPHA / 2)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(t_fracs * 100, obf_bounds, color="steelblue", linewidth=2.5,
        label="O'Brien-Fleming boundary")
ax.axhline(naive_bound, color="tomato", linewidth=1.8, linestyle="--",
           label=f"Naive fixed boundary (z = {naive_bound:.2f})")
ax.fill_between(t_fracs*100, obf_bounds, 6, alpha=0.08, color="steelblue",
                label="Rejection region (OBF)")
ax.set_xlabel("Information fraction — % of planned sample enrolled")
ax.set_ylabel("Critical z-value (reject if |z| > boundary)")
ax.set_title("O'Brien-Fleming: conservative early, standard at end")
ax.set_ylim(1.5, 5.5)
ax.legend()
plt.tight_layout()
plt.show()

print("OBF boundary at selected checkpoints:")
for t in [0.25, 0.50, 0.75, 1.00]:
    b = obrien_fleming_boundary(t, ALPHA)
    p_equiv = 2 * (1 - stats.norm.cdf(b))
    print(f"  t={t:.0%}  z_boundary={b:.3f}  (equivalent p < {p_equiv:.4f})")

In [ ]:
# mSPRT: always-valid likelihood ratio test
# The statistic Lambda_t > 1/alpha => reject at any time t
# Using normal approximation with mixing parameter tau

def msprt_lambda(z_t, n_t, tau=0.02):
    """
    Simplified normal mSPRT statistic.
    z_t: running z-statistic at sample size n_t (per group)
    tau: prior scale on effect size (smaller = more conservative)
    """
    v = 1.0 / n_t   # per-observation variance proxy
    rho = tau**2 / (tau**2 + v)
    return np.sqrt(1 - rho) * np.exp(0.5 * rho * z_t**2)

# Plot mSPRT statistic over the experiment using real data
df_sorted = df.sample(frac=1, random_state=42).reset_index(drop=True)
ctrl_s = df_sorted[df_sorted["group"]=="control"]["converted"].values
trt_s  = df_sorted[df_sorted["group"]=="treatment"]["converted"].values
min_n  = min(len(ctrl_s), len(trt_s))

steps = range(500, min_n, 500)
lambdas, obf_zs, naive_zs = [], [], []

for n in steps:
    c, t = ctrl_s[:n], trt_s[:n]
    p_pool = (c.sum() + t.sum()) / (2*n)
    se = np.sqrt(p_pool*(1-p_pool)*2/n)
    z = (t.mean() - c.mean()) / se if se > 0 else 0
    naive_zs.append(abs(z))
    obf_zs.append(obrien_fleming_boundary(n/min_n, ALPHA))
    lambdas.append(msprt_lambda(z, n))

steps_list = list(steps)
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 7), sharex=True)

ax1.plot(steps_list, naive_zs, color="gray", linewidth=1.2, label="|z| observed")
ax1.plot(steps_list, obf_zs,   color="steelblue", linewidth=2, label="OBF boundary")
ax1.axhline(naive_bound, color="tomato", linestyle="--", linewidth=1.5, label="Naive boundary")
ax1.set_ylabel("|z-statistic|")
ax1.set_title("Running z-statistic vs stopping boundaries")
ax1.legend(fontsize=9)

ax2.plot(steps_list, lambdas, color="coral", linewidth=2, label="mSPRT Lambda")
ax2.axhline(1/ALPHA, color="steelblue", linewidth=1.8, linestyle="--",
            label=f"Rejection threshold (1/alpha = {1/ALPHA:.0f})")
ax2.set_xlabel("Users enrolled (per group)")
ax2.set_ylabel("Lambda (mSPRT statistic)")
ax2.set_title("mSPRT: reject when Lambda > 1/alpha (valid at any time)")
ax2.legend(fontsize=9)

plt.tight_layout()
plt.show()

### Business Commentary

- **Spotify** adopted always-valid inference (Johari et al., 2022) as the default
  in their experimentation platform, replacing fixed-horizon tests entirely.
- **Airbnb** uses sequential testing to allow safe early stopping when an experiment
  shows strong harm to a guardrail metric.
- **Statsig** and **Eppo** both offer sequential testing as a first-class feature,
  which signals how widespread the adoption has become.

**O'Brien-Fleming vs mSPRT:**

| | O'Brien-Fleming | mSPRT |
|---|---|---|
| Requires pre-planned N | Yes | No |
| Works with continuous monitoring | No (fixed looks only) | Yes |
| Intuition | Spend alpha budget over time | Likelihood ratio martingale |
| Best for | Clinical trials, planned interim looks | Web experiments, dashboards |

**The key intuition:** standard p-values are not valid to check repeatedly.
Sequential methods are designed from the ground up to be checked at any time.
They achieve this by being more conservative early in the experiment.

---
## Summary: Standard vs Industry-Grade

| Dimension | Standard approach | Industry-grade |
|---|---|---|
| **Variance** | Raw metric variance | CUPED reduces it using prior data |
| **Ratio metrics** | Naive SE (wrong) | Delta method SE (correct) |
| **Monitoring** | Fixed end date, no peeking | Sequential testing, always valid |
| **Speed** | Fixed by traffic | CUPED can cut needed N by 20–50% |
| **Flexibility** | Commit upfront | Sequential allows early stopping |

These methods compound: a company using all three on a single experiment can get
results that are faster, more accurate, and continuously monitorable — without
sacrificing statistical validity.

Proceed to **[10 — Advanced Experiment Designs](10_advanced_designs.ipynb)**
for methods that handle situations where standard user-level randomization breaks.